Дане домашнє завдання буде повністю пов'язане з лінійною регресією та її реалізацією. <br/>
Отож розіб'ємо наше домашнє завдання на декілька частин:
- напишіть функцію гіпотези лінійної регресії у векторному вигляді;
- створіть функцію для обчислення функції втрат у векторному вигляді;
- реалізуйте один крок градієнтного спуску;
- знайдіть найкращі параметри w-> для датасету використовуючи написані вами функції, прогнозуючу ціну на будинок залежно від площі, кількості ванних кімнат та кількості спалень;
- знайдіть ці ж параметри за допомогою аналітичного рішення;
- для перевірки спрогнозованих значень, використайте LinearRegression з бібліотеки scikit-learn та порівняйте результати.

In [1406]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler

np.set_printoptions(precision=4)
rng = np.random.default_rng(seed=42)

## Підготовка датасету


In [1407]:
reader = pd.read_csv('datasets/Housing.csv', chunksize=10000)
df = pd.concat(reader, ignore_index=True)
df.drop_duplicates(inplace=True)

In [1408]:
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [1409]:
df.isna().sum().sort_values(ascending=False)

price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64

In [1410]:
df[['price', 'area']].describe()

,price,area
count,5.450000e+02,545.000000
mean,4.766729e+06,5150.541284
std,1.870440e+06,2170.141023
min,1.750000e+06,1650.000000
25%,3.430000e+06,3600.000000
50%,4.340000e+06,4600.000000
75%,5.740000e+06,6360.000000
max,1.330000e+07,16200.000000


In [1411]:
cols_to_convert = df.dtypes[(df.dtypes == 'str')].index
for column in cols_to_convert:
    uniques = df[column].unique().tolist()
    df[column] = df[column].map(lambda x: uniques.index(x))

df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,0,0,0,0,0,2,0,0
1,12250000,8960,4,4,4,0,0,0,0,0,3,1,0
2,12250000,9960,3,2,2,0,0,1,0,1,2,0,1
3,12215000,7500,4,2,2,0,0,1,0,0,3,0,0
4,11410000,7420,4,1,2,0,1,1,0,0,2,1,0


## Реалізація функцій регресії, втрат та градієнтного супску

In [1412]:
# функція лінійної регресії
def lin_req_func(x: np.ndarray, w: np.ndarray) -> np.ndarray:
    ones = np.ones((len(x), 1))
    extended_x = np.hstack((ones, x))

    return np.dot(extended_x, w)

In [1413]:
# функція втрат у векторному вигляді
def loss_func(predicted: np.ndarray, actual: np.ndarray) -> np.ndarray:
    return (1 / (2 * len(actual))) * np.sum(np.square(predicted - actual))

In [1414]:
# функція градієнтного спуску
def gradient_func(w: np.ndarray, predicted: np.ndarray, actual: np.ndarray, x: np.ndarray, rate: float) -> np.ndarray:
    m = len(actual)
    ones = np.ones((len(x), 1))
    extended_x = np.hstack((ones, x))

    grad = (1 / m) * ((predicted - actual) @ extended_x)
    return w - rate * grad

## Тестовий прогон власної реалізації лінійнох регресіх
Для тестування використаюмо згенеровані данні з лінійною залежністю та доданим шумом

In [1415]:
n_samples, n_features = 10000, 10

X = rng.uniform(low=1, high=10, size=(n_samples, n_features))
w = rng.random(n_features + 1)

noise = rng.normal(loc=0, scale=0.5, size=n_samples)
Y = w[0] + X @ w[1:] + noise

In [1416]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

prev_loss: float | None = None
loss_diff: float = float('inf')

while loss_diff > 1e-10:
    pred = lin_req_func(X_train, w.T)
    loss = loss_func(pred, y_train)
    w = gradient_func(w, pred, y_train, X_train, 0.0001)

    if prev_loss is not None:
        loss_diff = abs(loss - prev_loss)
    prev_loss = loss

    print(f'Loss: {loss:.12f} | Prev loss: {prev_loss:.12f} | Loss diff: {loss_diff:.12f}')


pred = lin_req_func(X_test, w.T)
print(f'Predicted: {pred[0:5]}')
print(f'Actual: {Y[0:5]}')

Loss: 0.125805032263 | Prev loss: 0.125805032263 | Loss diff: inf
Loss: 0.125804094875 | Prev loss: 0.125804094875 | Loss diff: 0.000000937388
Loss: 0.125803203019 | Prev loss: 0.125803203019 | Loss diff: 0.000000891856
Loss: 0.125802353917 | Prev loss: 0.125802353917 | Loss diff: 0.000000849101
Loss: 0.125801544964 | Prev loss: 0.125801544964 | Loss diff: 0.000000808953
Loss: 0.125800773714 | Prev loss: 0.125800773714 | Loss diff: 0.000000771251
Loss: 0.125800037869 | Prev loss: 0.125800037869 | Loss diff: 0.000000735845
Loss: 0.125799335274 | Prev loss: 0.125799335274 | Loss diff: 0.000000702595
Loss: 0.125798663906 | Prev loss: 0.125798663906 | Loss diff: 0.000000671368
Loss: 0.125798021866 | Prev loss: 0.125798021866 | Loss diff: 0.000000642041
Loss: 0.125797407370 | Prev loss: 0.125797407370 | Loss diff: 0.000000614496
Loss: 0.125796818745 | Prev loss: 0.125796818745 | Loss diff: 0.000000588625
Loss: 0.125796254419 | Prev loss: 0.125796254419 | Loss diff: 0.000000564325
Loss: 0.12

## Нормалізація та розбиття датасету

In [1417]:
features = ['area', 'bedrooms', 'bathrooms']
to_predict = ['price']

In [1418]:
df[[*to_predict, *features]].head()

,price,area,bedrooms,bathrooms
0,13300000,7420,4,2
1,12250000,8960,4,4
2,12250000,9960,3,2
3,12215000,7500,4,2
4,11410000,7420,4,1


In [1419]:
scaler = StandardScaler()
df_scaled = df[[*to_predict, *features]].copy()
df_scaled[[*to_predict, *features]] = scaler.fit_transform(df[[*to_predict, *features]])

df_scaled.head()

,price,area,bedrooms,bathrooms
0,4.566365,1.046726,1.403419,1.421812
1,4.004484,1.757010,1.403419,5.405809
2,4.004484,2.218232,0.047278,1.421812
3,3.985755,1.083624,1.403419,1.421812
4,3.554979,1.046726,1.403419,-0.570187


In [1420]:
X = df_scaled[features].to_numpy()
Y = df_scaled[to_predict].to_numpy().T[0]

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

## Моделювання за допомогою власної реалізації лінійної регресії

In [1421]:
w = rng.random((X_train.shape[1] + 1))

prev_loss: float | None = None
loss_diff: float = float('inf')

while loss_diff > 1e-10:
    pred = lin_req_func(X_train, w.T)
    loss = loss_func(pred, y_train)
    w = gradient_func(w, pred, y_train, X_train, 0.5)

    if prev_loss is not None:
        loss_diff = abs(loss - prev_loss)
    prev_loss = loss

    print(f'Loss: {loss:.12f} | Prev loss: {prev_loss:.12f} | Loss diff: {loss_diff:.12f}')

Loss: 0.943035372129 | Prev loss: 0.943035372129 | Loss diff: inf
Loss: 0.336924205236 | Prev loss: 0.336924205236 | Loss diff: 0.606111166893
Loss: 0.247969225696 | Prev loss: 0.247969225696 | Loss diff: 0.088954979540
Loss: 0.229529031628 | Prev loss: 0.229529031628 | Loss diff: 0.018440194068
Loss: 0.225046455100 | Prev loss: 0.225046455100 | Loss diff: 0.004482576529
Loss: 0.223890876774 | Prev loss: 0.223890876774 | Loss diff: 0.001155578325
Loss: 0.223583966876 | Prev loss: 0.223583966876 | Loss diff: 0.000306909898
Loss: 0.223500040566 | Prev loss: 0.223500040566 | Loss diff: 0.000083926310
Loss: 0.223476139214 | Prev loss: 0.223476139214 | Loss diff: 0.000023901352
Loss: 0.223468916543 | Prev loss: 0.223468916543 | Loss diff: 0.000007222672
Loss: 0.223466550461 | Prev loss: 0.223466550461 | Loss diff: 0.000002366081
Loss: 0.223465696895 | Prev loss: 0.223465696895 | Loss diff: 0.000000853566
Loss: 0.223465357416 | Prev loss: 0.223465357416 | Loss diff: 0.000000339479
Loss: 0.22

In [1422]:
y_pred = lin_req_func(X_test, w.T)
error = mean_squared_error(y_test, y_pred)
print(f'Mean squared error: {error:.12f}')

print(f'Coefficients: {w}')

Mean squared error: 0.787496369084
Coefficients: [-0.0163  0.4008  0.1421  0.3821]


In [1423]:
predicted_data = scaler.inverse_transform(np.hstack((y_pred.reshape(-1, 1), X_test)))
real_data = scaler.inverse_transform(np.hstack((y_test.reshape(-1, 1), X_test)))

predicted_df = pd.DataFrame(predicted_data, columns=['predicted_price', *features])
real_df = pd.DataFrame(real_data, columns=['real_price', *features])

real_df['predicted_price'] = predicted_df['predicted_price']
real_df[['predicted_price', 'real_price']].head()

,predicted_price,real_price
0,6.383163e+06,4060000.0
1,6.230229e+06,6650000.0
2,3.597876e+06,3710000.0
3,4.289739e+06,6440000.0
4,3.930453e+06,2800000.0


## Моедлювання за допомогою sklearn LinearRegression

In [1424]:
reg = sklearn.linear_model.LinearRegression()
reg.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](3,)","[0.4 ,0.14,0.38]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-0.01628
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,3
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(3)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](3,)","[25.41,19.82,15.87]"


In [1425]:
y_pred = reg.predict(X_test)
error = mean_squared_error(y_test, y_pred)
print(f'Mean squared error: {error:.12f}')

print(f'Coefficients: {reg.coef_}')
print(f'Intercept: {reg.intercept_:.4f}')

Mean squared error: 0.787497719214
Coefficients: [0.4008 0.1421 0.3821]
Intercept: -0.0163


In [1426]:
predicted_data = scaler.inverse_transform(np.hstack((y_pred.reshape(-1, 1), X_test)))
real_data = scaler.inverse_transform(np.hstack((y_test.reshape(-1, 1), X_test)))

predicted_df = pd.DataFrame(predicted_data, columns=['predicted_price',*features])
real_df = pd.DataFrame(real_data, columns=['real_price', *features])

real_df['predicted_price'] = predicted_df['predicted_price']
real_df[['predicted_price', 'real_price']].head()

,predicted_price,real_price
0,6.383168e+06,4060000.0
1,6.230250e+06,6650000.0
2,3.597885e+06,3710000.0
3,4.289731e+06,6440000.0
4,3.930446e+06,2800000.0


## Аналітичне рішення

In [1427]:
X_train_extended = np.hstack((np.ones((len(X_train), 1)), X_train))
w = np.linalg.solve(X_train_extended.T @ X_train_extended, X_train_extended.T @ y_train)

In [1428]:
y_pred = lin_req_func(X_test, w.T)
error = mean_squared_error(y_test, y_pred)
print(f'Mean squared error: {error:.12f}')

print(f'Coefficients: {reg.coef_}')
print(f'Intercept: {reg.intercept_:.4f}')

Mean squared error: 0.787497719214
Coefficients: [0.4008 0.1421 0.3821]
Intercept: -0.0163


In [1429]:
predicted_data = scaler.inverse_transform(np.hstack((y_pred.reshape(-1, 1), X_test)))
real_data = scaler.inverse_transform(np.hstack((y_test.reshape(-1, 1), X_test)))

predicted_df = pd.DataFrame(predicted_data, columns=['predicted_price',*features])
real_df = pd.DataFrame(real_data, columns=['real_price', *features])

real_df['predicted_price'] = predicted_df['predicted_price']
real_df[['predicted_price', 'real_price']].head()

,predicted_price,real_price
0,6.383168e+06,4060000.0
1,6.230250e+06,6650000.0
2,3.597885e+06,3710000.0
3,4.289731e+06,6440000.0
4,3.930446e+06,2800000.0
